# 02 - Descriptive Statistics, Correlation and Probability

This learning notebook covers **Part 1, Tasks 2 and 3** of the Smart City Traffic Intelligence capstone. It uses the verified SQLite database created in the previous lesson.

## Learning objectives

1. Calculate mean, median, standard deviation, variance and range.
2. Distinguish sample and population measures.
3. Calculate and interpret Pearson correlation.
4. Calculate joint and conditional probabilities from event counts.
5. Test independence using `P(A AND B) = P(A) × P(B)`.
6. Construct a 2×2 table and calculate an odds ratio.
7. Explain analytical limitations without making causal claims.

> This notebook is a learning and validation artifact. We will create the final written statistical analysis only after you run and understand these calculations.

## 1. Import tools and locate the verified database

Python's `statistics` module provides the required descriptive statistics and Pearson correlation. SQLite supplies the already-validated records.

In [1]:
import math
import sqlite3
import statistics
from pathlib import Path
from pprint import pprint

DATABASE_RELATIVE_PATH = Path("part1_data_analytics/sql/traffic_analysis.db")

def find_project_root(start: Path, relative_path: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / relative_path).is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {relative_path}. Run the Part 1 database builder first."
    )

PROJECT_ROOT = find_project_root(Path.cwd(), DATABASE_RELATIVE_PATH)
DATABASE_PATH = PROJECT_ROOT / DATABASE_RELATIVE_PATH

print(f"Database: {DATABASE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Database size: {DATABASE_PATH.stat().st_size:,} bytes")

Database: part1_data_analytics/sql/traffic_analysis.db
Database size: 6,234,112 bytes


## 2. Load only the fields required for these tasks

Selecting only necessary columns reduces memory use and makes the purpose of the analysis clear. The returned values are already numeric because the database schema uses `REAL` and `INTEGER` types.

In [2]:
with sqlite3.connect(DATABASE_PATH) as connection:
    records = connection.execute(
        """
        SELECT date_time, temp, weather_main, traffic_volume
        FROM traffic
        ORDER BY row_id
        """
    ).fetchall()

temperatures = [record[1] for record in records]
weather_conditions = [record[2] for record in records]
traffic_volumes = [record[3] for record in records]

print(f"Records loaded: {len(records):,}")
print(f"Temperature values: {len(temperatures):,}")
print(f"Traffic-volume values: {len(traffic_volumes):,}")

Records loaded: 48,204
Temperature values: 48,204
Traffic-volume values: 48,204


## 3. Calculate traffic-volume descriptive statistics

The assignment does not specify sample or population formulas. We report both transparently and use the **sample** standard deviation and variance as the primary values, matching common analytical-tool defaults.

- **Mean:** arithmetic average.
- **Median:** middle value after sorting.
- **Standard deviation:** typical spread around the mean, in vehicles.
- **Variance:** squared standard deviation, in squared vehicles.
- **Range:** maximum minus minimum.

In [3]:
traffic_statistics = {
    "count": len(traffic_volumes),
    "mean": statistics.fmean(traffic_volumes),
    "median": statistics.median(traffic_volumes),
    "sample_standard_deviation": statistics.stdev(traffic_volumes),
    "sample_variance": statistics.variance(traffic_volumes),
    "population_standard_deviation": statistics.pstdev(traffic_volumes),
    "population_variance": statistics.pvariance(traffic_volumes),
    "minimum": min(traffic_volumes),
    "maximum": max(traffic_volumes),
    "range": max(traffic_volumes) - min(traffic_volumes),
}

for name, value in traffic_statistics.items():
    if isinstance(value, float):
        print(f"{name:>32}: {value:,.4f}")
    else:
        print(f"{name:>32}: {value:,}")

                           count: 48,204
                            mean: 3,259.8184
                          median: 3,380.0000
       sample_standard_deviation: 1,986.8607
                 sample_variance: 3,947,615.3238
   population_standard_deviation: 1,986.8401
             population_variance: 3,947,533.4298
                         minimum: 0
                         maximum: 7,280
                           range: 7,280


### Interpretation

The mean and median are relatively close, but the standard deviation is large compared with the mean. Traffic demand therefore varies substantially across hours. This is expected for a corridor with overnight low demand and daytime commuting peaks. The range includes two zero-volume observations; those values require investigation before later cleaning, but they remain in this raw-data Part 1 calculation.

## 4. Check the effect of repeated timestamps

Some hours have multiple weather descriptions and therefore appear in several source rows with the same traffic volume. The one-record-per-hour calculation is a sensitivity check, not a silent replacement of the requested raw-data result.

In [4]:
with sqlite3.connect(DATABASE_PATH) as connection:
    hourly_traffic_volumes = [
        row[0]
        for row in connection.execute(
            """
            SELECT MAX(traffic_volume)
            FROM traffic
            GROUP BY date_time
            """
        )
    ]

hourly_sensitivity = {
    "unique_hours": len(hourly_traffic_volumes),
    "mean": statistics.fmean(hourly_traffic_volumes),
    "median": statistics.median(hourly_traffic_volumes),
    "sample_standard_deviation": statistics.stdev(hourly_traffic_volumes),
}

pprint(hourly_sensitivity, sort_dicts=False)
print(
    "Mean difference (hourly minus raw): "
    f"{hourly_sensitivity['mean'] - traffic_statistics['mean']:.4f}"
)

{'unique_hours': 40575,
 'mean': 3290.650474430068,
 'median': 3427,
 'sample_standard_deviation': 1984.7729092722152}
Mean difference (hourly minus raw): 30.8321


## 5. Calculate Pearson correlation

Pearson's `r` measures the direction and strength of a linear relationship. It ranges from `-1` to `+1`. This notebook uses the following stated interpretation guide:

- `|r| < 0.20`: very weak
- `0.20 ≤ |r| < 0.40`: weak
- `0.40 ≤ |r| < 0.60`: moderate
- `0.60 ≤ |r| < 0.80`: strong
- `|r| ≥ 0.80`: very strong

These labels are conventions, not universal scientific boundaries.

In [5]:
def describe_correlation(value: float) -> tuple[str, str]:
    direction = "positive" if value > 0 else "negative" if value < 0 else "none"
    magnitude = abs(value)
    if magnitude < 0.20:
        strength = "very weak"
    elif magnitude < 0.40:
        strength = "weak"
    elif magnitude < 0.60:
        strength = "moderate"
    elif magnitude < 0.80:
        strength = "strong"
    else:
        strength = "very strong"
    return direction, strength

raw_correlation = statistics.correlation(temperatures, traffic_volumes)
direction, strength = describe_correlation(raw_correlation)

valid_temperature_pairs = [
    (temperature, traffic)
    for temperature, traffic in zip(temperatures, traffic_volumes)
    if temperature > 0
]
cleaned_temperature_correlation = statistics.correlation(
    [pair[0] for pair in valid_temperature_pairs],
    [pair[1] for pair in valid_temperature_pairs],
)

print(f"Raw Pearson correlation: {raw_correlation:.6f}")
print(f"Direction: {direction}")
print(f"Strength using the stated guide: {strength}")
print(
    "Sensitivity excluding impossible 0 K readings: "
    f"{cleaned_temperature_correlation:.6f}"
)

Raw Pearson correlation: 0.130299
Direction: positive
Strength using the stated guide: very weak
Sensitivity excluding impossible 0 K readings: 0.132291


### Correlation interpretation

The coefficient is positive but very weak. Higher temperatures are associated with slightly higher traffic volume in this raw dataset, but temperature alone explains very little of the hourly variation.

Correlation does **not** establish causation. Hour of day, weekday, season, holidays, missing periods and commuting behaviour can influence both temperature and traffic. A causal claim would require a stronger research design and control of confounding variables.

## 6. Define the probability events

The assignment defines congestion as traffic volume above 5,500 vehicles. We must explicitly define ambiguous weather wording:

- `Congestion`: `traffic_volume > 5500`
- `Clear Weather`: `weather_main == 'Clear'`
- `Cloudy Weather`: `weather_main == 'Clouds'`
- `High Temperature`: `temp > 292 K`

Probabilities are calculated at the source-record level because each record contains a weather observation. Repeated timestamp reporting remains a limitation to disclose.

In [6]:
congestion = [volume > 5500 for volume in traffic_volumes]
clear_weather = [weather == "Clear" for weather in weather_conditions]
cloudy_weather = [weather == "Clouds" for weather in weather_conditions]
high_temperature = [temperature > 292 for temperature in temperatures]

record_count = len(records)
congestion_count = sum(congestion)
clear_count = sum(clear_weather)
cloudy_count = sum(cloudy_weather)
congestion_and_clear_count = sum(
    is_congested and is_clear
    for is_congested, is_clear in zip(congestion, clear_weather)
)
high_temperature_and_congestion_count = sum(
    is_hot and is_congested
    for is_hot, is_congested in zip(high_temperature, congestion)
)

event_counts = {
    "all_records": record_count,
    "congestion": congestion_count,
    "clear_weather": clear_count,
    "cloudy_weather": cloudy_count,
    "congestion_and_clear": congestion_and_clear_count,
    "high_temperature_and_congestion": high_temperature_and_congestion_count,
}
pprint(event_counts, sort_dicts=False)

{'all_records': 48204,
 'congestion': 7100,
 'clear_weather': 13391,
 'cloudy_weather': 15164,
 'congestion_and_clear': 1763,
 'high_temperature_and_congestion': 1867}


## 7. Calculate basic and conditional probabilities

For an event `A`, `P(A) = count(A) / count(all records)`. For conditional probability, `P(A | B) = count(A AND B) / count(B)`.

In [7]:
probability_congestion = congestion_count / record_count
probability_clear = clear_count / record_count
probability_congestion_and_clear = congestion_and_clear_count / record_count
probability_clear_given_congestion = (
    congestion_and_clear_count / congestion_count
)
probability_high_temperature_given_congestion = (
    high_temperature_and_congestion_count / congestion_count
)

probability_results = {
    "P(Congestion)": probability_congestion,
    "P(Clear Weather)": probability_clear,
    "P(Congestion AND Clear Weather)": probability_congestion_and_clear,
    "P(Clear Weather | Congestion)": probability_clear_given_congestion,
    "P(High Temperature | Congestion)": (
        probability_high_temperature_given_congestion
    ),
}

for name, value in probability_results.items():
    print(f"{name:<42} = {value:.6f} ({value:.2%})")

P(Congestion)                              = 0.147291 (14.73%)
P(Clear Weather)                           = 0.277799 (27.78%)
P(Congestion AND Clear Weather)            = 0.036574 (3.66%)
P(Clear Weather | Congestion)              = 0.248310 (24.83%)
P(High Temperature | Congestion)           = 0.262958 (26.30%)


## 8. Check congestion/clear-weather independence

Under the assignment rule, congestion and clear weather are independent only if:

`P(Congestion AND Clear) = P(Congestion) × P(Clear)`

Exact equality is the requested decision rule. In statistical practice, sampling uncertainty and effect size would also be considered.

In [8]:
independence_product = probability_congestion * probability_clear
independence_difference = (
    probability_congestion_and_clear - independence_product
)
events_are_independent = math.isclose(
    probability_congestion_and_clear,
    independence_product,
    rel_tol=0.0,
    abs_tol=1e-12,
)

print(f"Observed joint probability: {probability_congestion_and_clear:.6f}")
print(f"Independence product:       {independence_product:.6f}")
print(f"Difference:                 {independence_difference:.6f}")
print(f"Independent under equality rule: {events_are_independent}")

Observed joint probability: 0.036574
Independence product:       0.040917
Difference:                 -0.004343
Independent under equality rule: False


## 9. Calculate the clear-versus-cloudy congestion odds ratio

For each weather group:

`odds of congestion = congested records / non-congested records`

The odds ratio is:

`clear congestion odds / cloudy congestion odds`

An odds ratio below 1 means congestion odds are lower in clear records than in cloudy records.

In [9]:
clear_congested = sum(
    is_clear and is_congested
    for is_clear, is_congested in zip(clear_weather, congestion)
)
clear_not_congested = sum(
    is_clear and not is_congested
    for is_clear, is_congested in zip(clear_weather, congestion)
)
cloudy_congested = sum(
    is_cloudy and is_congested
    for is_cloudy, is_congested in zip(cloudy_weather, congestion)
)
cloudy_not_congested = sum(
    is_cloudy and not is_congested
    for is_cloudy, is_congested in zip(cloudy_weather, congestion)
)

clear_congestion_odds = clear_congested / clear_not_congested
cloudy_congestion_odds = cloudy_congested / cloudy_not_congested
odds_ratio = clear_congestion_odds / cloudy_congestion_odds
probability_congestion_given_clear = clear_congested / clear_count
probability_congestion_given_cloudy = cloudy_congested / cloudy_count

print("2 x 2 counts")
print(f"Clear  - congested: {clear_congested:,}; not congested: {clear_not_congested:,}")
print(f"Clouds - congested: {cloudy_congested:,}; not congested: {cloudy_not_congested:,}")
print()
print(f"P(Congestion | Clear):  {probability_congestion_given_clear:.2%}")
print(f"P(Congestion | Clouds): {probability_congestion_given_cloudy:.2%}")
print(f"Clear congestion odds:  {clear_congestion_odds:.6f}")
print(f"Cloudy congestion odds: {cloudy_congestion_odds:.6f}")
print(f"Odds ratio:              {odds_ratio:.6f}")

2 x 2 counts
Clear  - congested: 1,763; not congested: 11,628
Clouds - congested: 2,592; not congested: 12,572

P(Congestion | Clear):  13.17%
P(Congestion | Clouds): 17.09%
Clear congestion odds:  0.151617
Cloudy congestion odds: 0.206172
Odds ratio:              0.735388


### Probability interpretation

The equality check indicates that clear weather and congestion are not independent in these source records. The odds ratio is below 1, so recorded congestion odds are lower under `Clear` than under `Clouds`. This is an association, not evidence that clouds cause congestion. Time of day, weekday patterns, seasonal coverage, repeated weather reporting and other confounders may explain part of the difference.

## 10. Run automated checks

These assertions protect against transcription errors and confirm that this notebook is analysing the reviewed database.

In [10]:
assert traffic_statistics["count"] == 48_204
assert math.isclose(traffic_statistics["mean"], 3259.8183553232097)
assert traffic_statistics["median"] == 3380.0
assert math.isclose(
    traffic_statistics["sample_standard_deviation"],
    1986.8606704463587,
)
assert math.isclose(traffic_statistics["sample_variance"], 3947615.3237665542)
assert traffic_statistics["range"] == 7280
assert math.isclose(raw_correlation, 0.13029879817112516)
assert congestion_count == 7_100
assert clear_count == 13_391
assert congestion_and_clear_count == 1_763
assert math.isclose(probability_congestion, 0.1472906812712638)
assert math.isclose(probability_clear_given_congestion, 0.2483098591549296)
assert math.isclose(
    probability_high_temperature_given_congestion,
    0.26295774647887327,
)
assert events_are_independent is False
assert (clear_congested, clear_not_congested) == (1_763, 11_628)
assert (cloudy_congested, cloudy_not_congested) == (2_592, 12_572)
assert math.isclose(odds_ratio, 0.7353882125733118)

print(
    "PASS - Part 1 descriptive statistics, correlation and probability "
    "checks matched expected results."
)

PASS - Part 1 descriptive statistics, correlation and probability checks matched expected results.


## 11. Interpretation checkpoint

Before creating the final analysis, make sure you can explain:

1. Why the standard deviation shows high hourly variability.
2. Why the sample and population results are slightly different.
3. Why `r ≈ 0.13` is positive but very weak.
4. Why correlation cannot establish causation.
5. How conditional probability differs from joint probability.
6. Why the events fail the assignment's independence equality.
7. What an odds ratio of approximately `0.735` means—and does not mean.

### Stop point

Run all cells and confirm that the final code cell reports `PASS`. Save the notebook, then we will convert these verified results into the final statistical and probability analysis.